# Homework: Numerical Integration

## Problem 1: Implementing and Comparing Quadrature Rules

**(a)** Implement the composite trapezoidal rule and Simpson's rule with the following signatures:

In [1]:
import numpy as np
from scipy import integrate, stats

def trapezoidal(f, a, b, n):
    """Composite trapezoidal rule with n subintervals.

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of subintervals.

    Returns
    -------
    float
        Approximate integral value.
    """
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def simpsons(f, a, b, n):
    """Composite Simpson's rule with n subintervals (n must be even).

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of subintervals (must be even).

    Returns
    -------
    float
        Approximate integral value.
    """
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's rule")

    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return (h / 3) * (
        y[0] + y[-1] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-1:2])
    )

/Users/wenbinwu/miniforge3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Test both on $I = \int_0^1 e^{-x^2} dx$ with $n = 4, 16, 64, 256$. For each $n$, report the absolute error.

In [2]:
f = lambda x: np.exp(-x**2)
true_val, _ = integrate.quad(f, 0, 1)
ns = [4, 16, 64, 256]

trap_errors = []
simp_errors = []
print('Problem 1 results')
for n in ns:
    trap_est = trapezoidal(f, 0, 1, n)
    simp_est = simpsons(f, 0, 1, n)
    trap_err = abs(trap_est - true_val)
    simp_err = abs(simp_est - true_val)
    trap_errors.append(trap_err)
    simp_errors.append(simp_err)
    print(f'n={n:>3} | trap err={trap_err:.6e} | Simpson err={simp_err:.6e}')

Problem 1 results
n=  4 | trap err=3.840035e-03 | Simpson err=3.124698e-05
n= 16 | trap err=2.395360e-04 | Simpson err=1.246233e-07
n= 64 | trap err=1.496917e-05 | Simpson err=4.872454e-10
n=256 | trap err=9.355663e-07 | Simpson err=1.903366e-12


**(b)** For each method, verify the theoretical convergence rate by computing the ratio of consecutive errors as $n$ doubles. The trapezoidal rule should show a ratio near 4 (since error is $O(n^{-2})$) and Simpson's rule should show a ratio near 16 (since error is $O(n^{-4})$).

Answers:


Since the listed n values grow by a factor of 4, the observed ratios are about 16 for trapezoidal and 256 for Simpson. That matches second-order and fourth-order convergence after quadrupling n.

In [3]:
trap_ratios = [trap_errors[i] / trap_errors[i + 1] for i in range(len(trap_errors) - 1)]
simp_ratios = [simp_errors[i] / simp_errors[i + 1] for i in range(len(simp_errors) - 1)]
print('trap consecutive error ratios:', [round(r, 3) for r in trap_ratios])
print('Simpson consecutive error ratios:', [round(r, 3) for r in simp_ratios])


trap consecutive error ratios: [16.031, 16.002, 16.0]
Simpson consecutive error ratios: [250.731, 255.771, 255.991]


## Problem 2: Variance Reduction with Antithetic Variables

Standard Monte Carlo integration draws independent uniform samples to estimate an integral. Antithetic variables is a variance reduction technique that pairs each sample $U_i$ with its "mirror" $a + b - U_i$ on the interval $[a, b]$. When the integrand is monotone, these pairs are negatively correlated, which reduces the variance of the estimator.

**(a)** Implement a Monte Carlo integrator that uses antithetic variables. For each of $n$ uniform draws $U_i$ on $[a, b]$, compute both $f(U_i)$ and $f(a + b - U_i)$, then average the pair. The estimate is the mean of these $n$ pairwise averages (using $2n$ total function evaluations).

In [4]:
def mc_antithetic(f, a, b, n, seed=None):
    """Monte Carlo integration with antithetic variables on [a, b].

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of antithetic pairs (total function evaluations = 2n).
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    estimate : float
        Antithetic MC estimate of the integral.
    se : float
        Standard error of the estimate.
    """
    rng = np.random.default_rng(seed)
    u = rng.uniform(a, b, size=n)
    pair_means = 0.5 * (f(u) + f(a + b - u))
    estimate = (b - a) * np.mean(pair_means)
    se = (b - a) * np.std(pair_means, ddof=1) / np.sqrt(n)
    return estimate, se


Test on $I = \int_0^1 e^{-x^2} dx$ with $n = 500, 5000, 50000$ pairs (use `seed=42` with `numpy.random.default_rng`). For each $n$, report the estimate, standard error, and absolute error.

In [5]:
from scipy import integrate

f = lambda x: np.exp(-x**2)
true_val, _ = integrate.quad(f, 0, 1)

for n in [500, 5000, 50000]:
    est, se = mc_antithetic(f, 0, 1, n, seed=42)
    abs_err = abs(est - true_val)
    print(
        f"n={n:>5} | estimate={est:.6f} | SE={se:.6f} | abs. error={abs_err:.6e}"
    )


n=  500 | estimate=0.747281 | SE=0.001224 | abs. error=4.573430e-04
n= 5000 | estimate=0.746854 | SE=0.000400 | abs. error=2.952988e-05
n=50000 | estimate=0.746836 | SE=0.000127 | abs. error=1.202625e-05


**(b)** For a fair comparison, standard MC with the same computational budget uses $2n$ independent samples. For each value of $n$ above, also compute the standard MC estimate and SE using $2n$ samples (same seed). Report the variance reduction factor $\text{SE}_{\text{standard}}^2 / \text{SE}_{\text{antithetic}}^2$ for each $n$.

In [6]:
def mc_integrate(f, a, b, n, seed=None):
    rng = np.random.default_rng(seed)
    x = rng.uniform(a, b, size=n)
    values = f(x)
    estimate = (b - a) * np.mean(values)
    se = (b - a) * np.std(values, ddof=1) / np.sqrt(n)
    return estimate, se

f = lambda x: np.exp(-x**2)

for n in [500, 5000, 50000]:
    ant_est, ant_se = mc_antithetic(f, 0, 1, n, seed=42)
    std_est, std_se = mc_integrate(f, 0, 1, 2 * n, seed=42)
    vrf = (std_se ** 2) / (ant_se ** 2)

    print(
        f"n={n:>5} | "
        f"standard MC: est={std_est:.6f}, SE={std_se:.6f} | "
        f"antithetic: est={ant_est:.6f}, SE={ant_se:.6f} | "
        f"VRF={vrf:.2f}"
    )

n=  500 | standard MC: est=0.747999, SE=0.006402 | antithetic: est=0.747281, SE=0.001224 | VRF=27.33
n= 5000 | standard MC: est=0.748929, SE=0.002005 | antithetic: est=0.746854, SE=0.000400 | VRF=25.13
n=50000 | standard MC: est=0.746396, SE=0.000635 | antithetic: est=0.746836, SE=0.000127 | VRF=25.06


**(c)** Explain why antithetic variables reduce variance for the integrand $e^{-x^2}$ on $[0, 1]$. Describe a type of function for which antithetic variables would provide little or no variance reduction.

Variance falls because $exp^{-x^2}$ is smooth and decreasing on $[0,1]$, so $f(U)$ and $f(1-U)$ are negatively correlated. Little benefit is expected for nearly constant, highly oscillatory, or symmetry-breaking functions where the mirrored pair is not negatively correlated.

## Problem 3: Monte Carlo Integration

**(a)** Write a function that estimates $\int_a^b f(x) dx$ by Monte Carlo integration and returns both the estimate and its standard error:

In [7]:
def mc_integrate(f, a, b, n, seed=None):
    """Monte Carlo integration on [a, b].

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of random samples.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    estimate : float
        MC estimate of the integral.
    se : float
        Standard error of the estimate.
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(a, b, size=n)
    values = f(x)
    estimate = (b - a) * np.mean(values)
    se = (b - a) * np.std(values, ddof=1) / np.sqrt(n)
    return estimate, se

Test on $\int_0^1 e^{-x^2} dx$ with $n = 100, 1000, 10000, 100000$ (use `seed=42`). For each $n$, report the estimate, standard error, and absolute error.


In [8]:
mc_results = {}
for n in [100, 1000, 10000, 100000]:
    est, se = mc_integrate(f, 0, 1, n, seed=42)
    mc_results[n] = (est, se, abs(est - true_val))
    print(f'P3 n={n:>6} | estimate={est:.6f} | se={se:.6f} | abs err={abs(est - true_val):.6f}')


P3 n=   100 | estimate=0.758353 | se=0.018883 | abs err=0.011529
P3 n=  1000 | estimate=0.747999 | se=0.006402 | abs err=0.001175
P3 n= 10000 | estimate=0.748929 | se=0.002005 | abs err=0.002105
P3 n=100000 | estimate=0.746396 | se=0.000635 | abs err=0.000428


**(b)** Verify that the MC error rate is $O(n^{-1/2})$: compute the ratio of standard errors as $n$ increases by a factor of 100 (from 100 to 10000). The SE should decrease by a factor of approximately 10.

In [9]:
print('SE ratio n=100 to n=10000:', mc_results[100][1] / mc_results[10000][1])

SE ratio n=100 to n=10000: 9.41790387833729


**(c)** The $O(n^{-1/2})$ rate appears slow compared to deterministic quadrature. With 256 subintervals, Simpson's rule achieves error around $10^{-12}$ for this 1D integral, while MC with 100,000 points still has error around $10^{-4}$. Why then is MC preferred for high-dimensional integrals?

Answer:
MC is still preferred in high dimensions because its convergence rate does not worsen with dimension, while deterministic grid-based quadrature suffers from the curse of dimensionality. If we use $n$ points per dimension in $d$ dimensions, product quadrature needs $n^d$ evaluations, which becomes infeasible very quickly. MC is slower in 1D, but much more scalable in large dimensions.

(MC with $n^d$ samples matches the computational cost of product quadrature, but replaces the deterministic tensor grid and quadrature weights with random sampling and averaging.)

## Problem 4: Importance Sampling

The following code estimates $P(Z > 4)$ where $Z \sim N(0,1)$ using naive Monte Carlo and importance sampling with a shifted exponential proposal. Read the code and answer the questions.

In [10]:
import numpy as np
from scipy import stats

true_prob = 1 - stats.norm.cdf(4)  # ≈ 3.17e-5

# Naive MC
np.random.seed(42)
z_samples = np.random.normal(0, 1, 100000)
naive_est = np.mean(z_samples > 4)
naive_count = np.sum(z_samples > 4)

# Importance sampling with Exp(1) shifted to start at 4
np.random.seed(42)
x_is = np.random.exponential(1.0, 100000) + 4.0
log_weights = stats.norm.logpdf(x_is) - (-1.0 * (x_is - 4))
weights = np.exp(log_weights)
is_est = np.mean(weights)

**(a)** Explain why naive MC performs poorly for this problem. How many of the 100,000 standard normal samples are expected to exceed 4?

Answer:

Naive MC performs poorly because P(Z > 4) is only about 3.17e-05, so 100000 draws produce about 3.17 exceedances on average. With this seed there was only 1 exceedance.

In [11]:
print('true probability:', true_prob)
print('naive exceedances in 100000 draws:', naive_count)
print('naive estimate:', naive_est)
print('importance sampling estimate:', is_est)

true probability: 3.167124183311998e-05
naive exceedances in 100000 draws: 1
naive estimate: 1e-05
importance sampling estimate: 3.1702562560108844e-05


**(b)** The importance sampling proposal is $q(x) = e^{-(x-4)}$ for $x \geq 4$. Verify that `log_weights` correctly computes $\log[\phi(x) / q(x)]$ where $\phi$ is the standard normal density.

Answer:
$q(x) = exp(-(x-4))$ for $x \geq 4$, so $log q(x) = -(x-4)$. Therefore log(phi(x)/q(x)) = stats.norm.logpdf(x) - (-(x-4)), matching the code.

**(c)** A classmate suggests using a $t$-distribution with 3 degrees of freedom as the proposal instead of the shifted exponential. Would this be a good choice for estimating $P(Z > 4)$? Why or why not?

Answer:

No. A t distribution with df=3 is bad because it is much heavier-tailed than the normal target tail, so it wastes mass far out in the tail and yields more variable weights.

**(d)** Another classmate proposes using $N(5, 0.5^2)$ as the proposal. Compute the effective sample size (ESS) and compare it to using the shifted exponential. Which proposal is better and why?

Answer:

The shifted exponential is better because it matches the support x >= 4 and gives larger ESS.



In [12]:
np.random.seed(42)
n_is = 10000

# Shifted exponential proposal (all samples are >= 4 by construction)
x_exp = np.random.exponential(1.0, n_is) + 4.0
log_w_exp = stats.norm.logpdf(x_exp) + (x_exp - 4)
log_w_exp -= np.max(log_w_exp)
w_exp = np.exp(log_w_exp)
w_exp_norm = w_exp / np.sum(w_exp)
ess_exp = 1.0 / np.sum(w_exp_norm ** 2)

# N(5, 0.5^2) proposal: multiply weights by indicator I(x > 4)
x_norm = np.random.normal(5, 0.5, n_is)
log_w_norm = stats.norm.logpdf(x_norm) - stats.norm.logpdf(x_norm, 5, 0.5)
w_norm_raw = np.exp(log_w_norm) * (x_norm > 4)
w_norm_raw[w_norm_raw > 0] /= np.max(w_norm_raw[w_norm_raw > 0])
w_norm_norm = w_norm_raw / np.sum(w_norm_raw)
ess_norm = 1.0 / np.sum(w_norm_norm ** 2)

print('ESS shifted exponential:', ess_exp)
print('ESS N(5, 0.5^2):', ess_norm)


ESS shifted exponential: 4137.000460254567
ESS N(5, 0.5^2): 677.4226510447007


## Problem 5: The Curse of Dimensionality

Deterministic quadrature rules extend to multiple dimensions via product grids. For a $d$-dimensional integral with $n$ nodes per dimension, the total number of function evaluations is $n^d$. Monte Carlo, by contrast, uses a fixed sample size regardless of dimension. In this problem, you will observe the crossover point where MC becomes more efficient than product-rule quadrature.

Consider the integral $I_d = \int_{[0,1]^d} e^{-\|\mathbf{x}\|^2} \, d\mathbf{x}$ where $\|\mathbf{x}\|^2 = x_1^2 + \cdots + x_d^2$.

**(a)** Write a function that computes $I_d$ using a product trapezoidal rule with $n$ nodes per dimension.

In [13]:
def trap_nd(f, d, n):
    """Product trapezoidal rule on [0,1]^d with n nodes per dimension.

    Parameters
    ----------
    f : callable
        Function from R^d to R. Takes an array of shape (N, d)
        where N is the number of evaluation points.
    d : int
        Number of dimensions.
    n : int
        Number of nodes per dimension.

    Returns
    -------
    float
        Approximate integral value.
    """
    grid = np.linspace(0, 1, n)
    h = 1 / (n - 1)
    meshes = np.meshgrid(*([grid] * d), indexing='ij')
    points = np.stack([m.ravel() for m in meshes], axis=1)
    values = f(points).reshape([n] * d)

    weights = np.ones([n] * d)
    for axis in range(d):
        left = [slice(None)] * d
        right = [slice(None)] * d
        left[axis] = 0
        right[axis] = -1
        weights[tuple(left)] *= 0.5
        weights[tuple(right)] *= 0.5

    return (h**d) * np.sum(weights * values)

def f_nd(x):
    return np.exp(-np.sum(x**2, axis=1))

The integrand separates as $e^{-\|\mathbf{x}\|^2} = \prod_{j=1}^d e^{-x_j^2}$, so the exact value is $I_d = \left(\int_0^1 e^{-x^2} dx\right)^d$. Use `scipy.integrate.quad` to compute the 1D reference value, then raise it to the $d$-th power. Compute the product trapezoidal estimate for $d = 1, 2, 3, 4, 5, 6$ with $n = 10$ nodes per dimension. For each $d$, report the number of function evaluations and the absolute error.

**(b)** For each dimension $d$ above, estimate $I_d$ using Monte Carlo with the same number of function evaluations as the product trapezoidal rule (i.e., $n^d$ MC samples). Use `seed=42` with `numpy.random.default_rng`. Compare the MC error to the trapezoidal error for each $d$. At what dimension does MC become more accurate?


In [14]:
one_d_true, _ = integrate.quad(lambda x: np.exp(-x**2), 0, 1)
for d in range(1, 7):
    m = 10**d
    exact = one_d_true**d
    trap_est = trap_nd(f_nd, d, 10)
    rng = np.random.default_rng(42)
    samples = rng.uniform(0, 1, size=(m, d))
    mc_vals = f_nd(samples)
    mc_est = np.mean(mc_vals)
    trap_err = abs(trap_est - exact)
    mc_err = abs(mc_est - exact)
    print(f'd={d} | evals={m} | trap err={trap_err:.6e} | mc err={mc_err:.6e}')


d=1 | evals=10 | trap err=7.572649e-04 | mc err=7.240726e-02
d=2 | evals=100 | trap err=1.130514e-03 | mc err=1.006530e-02
d=3 | evals=1000 | trap err=1.265801e-03 | mc err=8.350925e-04
d=4 | evals=10000 | trap err=1.259802e-03 | mc err=1.260478e-03
d=5 | evals=100000 | trap err=1.175467e-03 | mc err=6.159358e-04
d=6 | evals=1000000 | trap err=1.052907e-03 | mc err=1.016048e-06


MC first becomes more accurate at d=3 for this example.


**(c)** The separability $e^{-\|\mathbf{x}\|^2} = \prod_j e^{-x_j^2}$ means one could compute the $d$-dimensional integral as a product of $d$ one-dimensional integrals, each requiring only $n$ nodes, for a total of $n \cdot d$ evaluations instead of $n^d$. Explain why this trick does not generalize: give an example of a non-separable integrand on $[0,1]^d$ that cannot be decomposed this way, and explain why such integrands are common in statistical applications involving correlated random effects.

Answer:

The integral is separable when it can be written as a product of one-dimensional functions, one for each coordinate. 
A non-separable example is $e^{-(x_1 + ... + x_d)^2}$. Such kind of dependence is common in models with correlated random effects -- e.g., multivariate gaussian likelihood with a non-digonal covariance matrix.
